Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install Libraries

In [2]:
!pip install -q google-generativeai pandas openpyxl pillow

Imports

In [3]:
import os
import json
import random
import pandas as pd

from PIL import Image

import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Load Gemini API

In [4]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

Dataset Path

In [5]:
dataset_path = "/content/drive/MyDrive/Medical_Report_Extraction_Project/Dataset_OCR"

images = [
    f for f in os.listdir(dataset_path)
    if f.lower().endswith(".jpg")
]

print("Total Images:", len(images))

Total Images: 100


Random Selection (Training on 5 images and Testing on 2 images)

In [6]:
random.seed(842)

selected_images = random.sample(images, 7)

training_images = selected_images[:5]
testing_images = selected_images[5:]

print("Training Images:")
print(training_images)

print("\nTesting Images:")
print(testing_images)

Training Images:
['thyrocare_0_9047.jpg', 'thyrocare_0_7219.jpg', 'thyrocare_0_3753.jpg', 'thyrocare_0_8251.jpg', 'thyrocare_0_6698.jpg']

Testing Images:
['thyrocare_0_7109.jpg', 'thyrocare_0_912.jpg']


Process Testing Images

In [7]:
records = []

for file in testing_images:

    image_path = os.path.join(
        dataset_path,
        file
    )

    img = Image.open(image_path)

    prompt = """
You are a medical report information extraction assistant.

Analyze this medical report image.

Extract:

1. name
2. test_asked
3. test_name
4. technology
5. value
6. unit
7. clinical_significance

Return ONLY valid JSON.

Example:

{
  "name":"",
  "test_asked":"",
  "test_name":"",
  "technology":"",
  "value":"",
  "unit":"",
  "clinical_significance":""
}

Rules:
- Do not return markdown.
- Do not return explanations.
- If a field is missing, return "Not Found".
"""

    response = model.generate_content(
            [prompt, img]
        )

    result = response.text.strip()
    result = result.replace("```json","")
    result = result.replace("```","")

    try:

        data = json.loads(result)

        data["image_name"] = file

        data["dataset_type"] = (
            "Training"
            if file in training_images
            else "Testing"
        )

        records.append(data)

        print(f"Processed: {file}")

    except Exception as e:

        print(f"Error: {file}")
        print(e)

Processed: thyrocare_0_7109.jpg
Processed: thyrocare_0_912.jpg


Create DataFrame

In [8]:
df = pd.DataFrame(records)

df

,name,test_asked,test_name,technology,value,unit,clinical_significance,image_name,dataset_type
0,ASHIM SENGUPTA (58Y/M),AAROGYAM C,ALKALINE PHOSPHATASE,PHOTOMETRY,67,U/L,Please correlate with clinical conditions.,thyrocare_0_7109.jpg,Testing
1,ASHIM KR SENGUPTA (25Y/M),AAROGYAM A,TOTAL TRIIODOTHYRONINE (T3),C.L.I.A,76,ng/dl,SUGGESTING THYRONORMALCY,thyrocare_0_912.jpg,Testing


Save Output Files

In [9]:

output_folder = "/content/drive/MyDrive/Medical_Report_Extraction_Project/output"

os.makedirs(output_folder, exist_ok=True)

csv_path = f"{output_folder}/reports.csv"
excel_path = f"{output_folder}/reports.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(excel_path, index=False)

print("CSV Saved:", csv_path)
print("Excel Saved:", excel_path)

CSV Saved: /content/drive/MyDrive/Medical_Report_Extraction_Project/output/reports.csv
Excel Saved: /content/drive/MyDrive/Medical_Report_Extraction_Project/output/reports.xlsx


Verify Output

In [14]:
df.head()

,name,test_asked,test_name,technology,value,unit,clinical_significance,image_name,dataset_type
0,ASHIM SENGUPTA (58Y/M),AAROGYAM C,ALKALINE PHOSPHATASE,PHOTOMETRY,67,U/L,Please correlate with clinical conditions.,thyrocare_0_7109.jpg,Testing
1,ASHIM KR SENGUPTA (25Y/M),AAROGYAM A,TOTAL TRIIODOTHYRONINE (T3),C.L.I.A,76,ng/dl,SUGGESTING THYRONORMALCY,thyrocare_0_912.jpg,Testing
